# 16 SCI Readiness Pack

Run this notebook after stage 15 completes. It creates paper-facing effect-size, profile-ablation, and claim-readiness artifacts.


In [ ]:
from pathlib import Path
import os, sys

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(base: Path) -> bool:
    return (base / "pyproject.toml").exists() and (base / "src" / "cybsage_marl").exists()


def _mount_drive_if_colab():
    if not Path("/content").exists():
        return
    if Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive auto-mount failed:", exc)


def _find_repo_root():
    _mount_drive_if_colab()
    candidates = []
    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    if os.environ.get("CYBSAGE_REPO_ROOT"):
        candidates.append(Path(os.environ["CYBSAGE_REPO_ROOT"]).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content/drive/MyDrive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/IMPLEMENTATION"),
        Path("/content/drive/My Drive/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/Volumes/LocalDrive1/onlyscholar-projects/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
    ])

    checked = []
    seen = set()
    for base in candidates:
        base = base.expanduser()
        if str(base) in seen:
            continue
        seen.add(str(base))
        checked.append(base)
        if _looks_like_repo(base):
            return base

    # Last-resort bounded search. This catches Drive shortcuts/copies while avoiding a full Drive crawl.
    search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive"), Path("/content")]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for pyproject in root.glob("**/pyproject.toml"):
                base = pyproject.parent
                checked.append(base)
                if base.name == "IMPLEMENTATION" and _looks_like_repo(base):
                    return base
        except Exception as exc:
            checked.append(Path(f"{root} [search failed: {exc}]"))

    raise RuntimeError(
        "Could not find CybSAGE IMPLEMENTATION repo root visible to this runtime.\n\n"
        "Most common fix: in Colab, run drive.mount('/content/drive') and confirm the folder exists under "
        "/content/drive/MyDrive/projects/implementation/IMPLEMENTATION.\n\n"
        "Checked:\n" + "\n".join(map(str, checked[-80:]))
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Repository root:", REPO_ROOT)


In [ ]:
!PYTHONPATH=src python scripts/run_stage.py cage4_sci_readiness --sci-min-paired-runs 30


In [ ]:
import json, pandas as pd
from pathlib import Path
base = Path("outputs/16_sci_readiness_pack")
summary = json.loads((base / "sci_readiness_summary.json").read_text())
print(json.dumps(summary, indent=2))
effect = base / "cage4_effect_sizes.csv"
pd.read_csv(effect) if effect.exists() and effect.stat().st_size else "No effect table produced"


In [ ]:
from pathlib import Path
report = Path("outputs/16_sci_readiness_pack/sci_readiness_report.md")
print(report.read_text()[:6000] if report.exists() else "No report produced")
